# 🧠 Session 11 — Memory, Tools & Data Layer

## 🎯 Overview

In this session, we’ll explore how modern AI agents store, retrieve, and reason over knowledge — focusing on **memory**, **tools**, and **data integration** through **vector databases**.

You’ll learn to:
- Represent text as numerical embeddings for semantic search
- Store and retrieve knowledge from a vector database like **Qdrant**
- Extract structured outputs using **JSON Schema** and **Pydantic models**
- Integrate **memory** into agent workflows using **LangGraph** and **CrewAI**
- Build a **Legal Case-Law Assistant** that can store rulings and recall them with citations

This lecture is concept-heavy and implementation-driven — by the end, you’ll have a working intelligent retrieval system with structured reasoning.


## 🧩 1️⃣ The Data Layer in LLM Systems

Large Language Models don’t “remember” facts permanently — they only predict text.
The **data layer** bridges this gap by connecting LLMs to persistent external knowledge.

A typical agent’s data layer consists of:
1. **Embeddings Model** → converts text to numeric vectors
2. **Vector Store** → stores those embeddings for semantic retrieval
3. **Loader & Splitter** → ingest and chunk large documents
4. **Retriever** → finds semantically similar text to a user query
5. **LLM + Output Parser** → interprets results and produces structured answers

Together, these enable an LLM to *“think with memory”* instead of starting fresh each time.


## 🧮 2️⃣ Embeddings — Turning Text into Meaningful Numbers

**Embeddings** are dense numerical representations of text where similar meanings lie close in vector space.

- “contract breach” → `[0.12, 0.48, -0.33, …]`
- “violation of agreement” → `[0.13, 0.46, -0.31, …]`

When you compute the cosine similarity between these vectors, it’s high — meaning they are conceptually similar.

Common embedding providers:
| Provider | Notes |
|-----------|-------|
| **OpenAI** (`text-embedding-3-small` / `text-embedding-3-large`) | Best general-purpose quality |
| **VoyageAI** | Excellent for academic/legal texts |
| **Hugging Face (BGE, Instructor models)** | Open-source and local |

👉 In practice: You embed both your **documents** and your **query**, then retrieve the most similar ones.


## 🧭 3️⃣ Vector Databases — Memory for LLMs

A **Vector Database (VDB)** stores document embeddings for semantic search and reasoning.

Each record typically contains:
- `id` → unique identifier
- `vector` → embedding array
- `metadata` → e.g. title, source, timestamp
- `payload` → original text chunk

Popular Vector DBs:
| DB | Strengths |
|----|------------|
| **Chroma** | Local, simple, great for prototypes |
| **Qdrant** | Open-source, scalable, high-performance |
| **Pinecone** | Fully managed, easy scaling |
| **Weaviate** | Hybrid (vector + keyword), GraphQL interface |

For this session, we’ll use **Qdrant** because it’s production-ready yet easy to run locally.


## ⚙️ 4️⃣ Setting Up the Environment

We’ll use:
- `LangChain` for pipeline orchestration
- `OpenAIEmbeddings` for embeddings
- `Qdrant` for vector storage
- `Pydantic` for structured outputs

Make sure you have your API key set in the environment (`OPENAI_API_KEY`).


In [9]:
# 🧰 Install dependencies
!pip install langchain-openai langchain qdrant-client chromadb tiktoken pydantic langchain_community sentence-transformers


  Using cached torch-2.8.0-cp313-cp313-win_amd64.whl.metadata (30 kB)
  Using cached scikit_learn-1.7.2-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.16.2-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached pillow-11.3.0-cp313-cp313-win_amd64.whl.metadata (9.2 kB)
  Using cached safetensors-0.6.2-cp38-abi3-win_amd64.whl.metadata (4.1 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
    --------------------------------------- 0.3/12.0 MB ? eta -:--:--
    --------------------------------------- 0.3/12.0 MB ? eta -:--:--
    --------------------------------------- 0.3/12.0 MB ? eta -:--:--
    --------------------------------------- 0.3/12.0 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.0 MB 562.7 kB/s eta 0:00:20
   --- ---


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Qdrant
from langchain_community.document_loaders import TextLoader
from langchain_community.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
import qdrant_client


## 📚 5️⃣ Loading and Preparing Documents

Legal documents are usually long and complex — judgments, rulings, clauses, etc.
Before embedding, they need to be **loaded and split** into manageable pieces so the model can process them.

Chunking helps the retriever return meaningful, self-contained passages.


In [11]:
# Load a text file containing a legal ruling
loader = TextLoader("ruling.txt")
docs = loader.load()

# Split the document into smaller overlapping chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(docs)

print(f"📄 Loaded {len(chunks)} chunks.")


📄 Loaded 3 chunks.


## 💾 6️⃣ Embedding and Storing in Qdrant

We now convert each chunk into an embedding vector and push it into Qdrant.
For demonstration, we’ll run Qdrant in-memory; in production, you’d run a persistent server (`localhost:6333` or cloud instance).


In [12]:
from langchain_community.embeddings import HuggingFaceEmbeddings
# Initialize an in-memory Qdrant client
client = qdrant_client.QdrantClient(":memory:")

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a new collection and store vectors
vectorstore = Qdrant.from_documents(
    documents=chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="legal_rulings"
)
print("✅ Stored document chunks in Qdrant.")


C:\Users\Moonwalking\PycharmProjects\Session_11\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Stored document chunks in Qdrant.


## 🔍 7️⃣ Semantic Retrieval

Once embeddings are stored, we can search semantically — e.g.,
“cases about breach of contract and damages” will find relevant rulings even if those exact words aren’t present.


In [13]:
query = "precedents about contract breach and damages"
results = vectorstore.similarity_search(query, k=3)

for i, res in enumerate(results, 1):
    print(f"📘 Result {i}:")
    print(res.page_content[:300], "\n---\n")


📘 Result 1:
---

**Issues for Determination**

1. Whether Mason Technologies breached the contract by failing to meet delivery deadlines.
2. Whether Omnicorp's actions contributed to the delay by withholding necessary technical data.
3. The extent to which either party is entitled to damages.

---

**Findings** 
---

📘 Result 2:
---

**Ruling**

Justice Graves held that Omnicorp Industries materially breached the contract by failing to provide the required specifications in due time. The Plaintiff was awarded compensatory damages equivalent to the projectâ€™s remaining milestone payments, amounting to USD 250,000.

---

**C 
---

📘 Result 3:
IN THE HIGH COURT OF METROPOLIS

Case No. 2022/CV/1453
Between:
MASON TECHNOLOGIES LTD. (Plaintiff)
and
OMNICORP INDUSTRIES (Defendant)

Delivered on: 14 March 2022
Presiding Judge: Justice Eleanor Graves

---

**Background**

The Plaintiff, Mason Technologies Ltd., entered into a software developme 
---



## 🧱 8️⃣ Structured Output with Pydantic

Unstructured text responses are hard to store or analyze.
We can enforce structure using **Pydantic models**, defining a schema the LLM must follow.

This ensures every summary follows the same JSON format — ready for storage, retrieval, or display.


In [14]:
class LegalSummary(BaseModel):
    case_name: str = Field(..., description="Case title or citation")
    issue: str = Field(..., description="Key legal issue")
    ruling: str = Field(..., description="Court's decision")
    citations: list[str] = Field(default_factory=list)


## 🤖 9️⃣ LLM Summarizer with Structured Output

We’ll use the model to produce a structured JSON summary of a ruling.

This approach:
- Keeps outputs machine-readable
- Enforces predictable fields
- Makes later indexing or comparison easy


In [17]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key="open router api key"
)


prompt = """You are a legal summarizer.
Extract structured legal information from the text below.
Return valid JSON following this schema:

{{
  "case_name": "string",
  "facts": "string",
  "ruling": "string",
  "citations": ["list of strings"]
}}

Text:
{case_text}
"""


response = llm.invoke(prompt.format(case_text=results[0].page_content))
print(response.content)


```json
{
  "case_name": "Mason Technologies v. Omnicorp",
  "facts": "Mason Technologies sought necessary technical documentation from Omnicorp to meet delivery deadlines. Evidence indicated that Omnicorp's engineering department delayed responses for over three months, hindering Mason's ability to fulfill contractual obligations.",
  "ruling": "The Court ruled that Mason Technologies was justified in its inability to meet delivery deadlines due to Omnicorp's failure to cooperate and provide necessary information.",
  "citations": []
}
```


## 🧠 🔁 1️⃣0️⃣ Adding Memory to Agents

**Memory** allows agents to remember prior steps, retrieved results, and user interactions.

Two main approaches:
1. **LangGraph Memory Nodes** → memory stored in graph state
2. **CrewAI Memory Components** → pluggable memories (conversation, vector, or summary)

### Example Memory Concepts
- **Conversation Memory:** remembers dialogue context
- **Vector Memory:** remembers facts semantically (like Qdrant chunks)
- **Summary Memory:** compresses long interactions into concise notes

This lets agents “recall” earlier cases or user inputs even after multiple turns.


## ⚖️ 1️⃣1️⃣ Case Study — Legal Case-Law Assistant

We’ll combine everything into a retrieval-augmented reasoning pipeline.

**Goal:**
Build an AI assistant that:
1. Ingests legal documents (rulings, precedents, case summaries)
2. Stores them as embeddings in Qdrant
3. Retrieves relevant rulings when a query arrives
4. Produces structured summaries with citations
5. Maintains conversation context across turns

### Workflow
1. **User uploads legal files** → loader reads them
2. **Chunk & embed** → store vectors in Qdrant
3. **Query** → find most similar cases
4. **LLM summarizer** → format structured answers
5. **Memory layer** → maintain ongoing context


In [18]:
def legal_assistant(query: str):
    """Simple pipeline for a case-law retrieval assistant."""
    retrieved_docs = vectorstore.similarity_search(query, k=2)
    combined_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

    response = llm.invoke(prompt.format(case_text=combined_text))
    print("🧾 Structured Summary:\n", response.content)
    return response

# Example usage
legal_assistant("cases about negligence in corporate contracts")


🧾 Structured Summary:
 ```json
{
  "case_name": "Mason Technologies Ltd. v. Omnicorp Industries",
  "facts": "Mason Technologies consistently communicated with Omnicorp seeking necessary technical documentation, but Omnicorp's engineering department delayed responses for over three months, preventing Mason from fulfilling its contractual obligations.",
  "ruling": "Justice Graves held that Omnicorp Industries materially breached the contract by failing to provide the required specifications in due time. The Plaintiff was awarded compensatory damages equivalent to the project’s remaining milestone payments, amounting to USD 250,000.",
  "citations": [
    "Anderson v. Stellar Logistics (2015) 2 MLJ 441",
    "Quantum Systems v. Rivera Holdings (2018) 4 MLR 209"
  ]
}
```


AIMessage(content='```json\n{\n  "case_name": "Mason Technologies Ltd. v. Omnicorp Industries",\n  "facts": "Mason Technologies consistently communicated with Omnicorp seeking necessary technical documentation, but Omnicorp\'s engineering department delayed responses for over three months, preventing Mason from fulfilling its contractual obligations.",\n  "ruling": "Justice Graves held that Omnicorp Industries materially breached the contract by failing to provide the required specifications in due time. The Plaintiff was awarded compensatory damages equivalent to the project’s remaining milestone payments, amounting to USD 250,000.",\n  "citations": [\n    "Anderson v. Stellar Logistics (2015) 2 MLJ 441",\n    "Quantum Systems v. Rivera Holdings (2018) 4 MLR 209"\n  ]\n}\n```', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 386, 'total_tokens': 551, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'au

## 🚀 1️⃣2️⃣ Next Steps & Extensions

Now that you have a working data layer and structured retrieval setup, you can extend this system by:

- **Integrating VoyageAI embeddings** for legal/academic domains
- **Persisting Qdrant collections** on disk or remote cluster
- **Adding metadata filtering** (e.g., by jurisdiction or year)
- **Incorporating LangGraph memory nodes** for ongoing chat sessions
- **Linking to external data sources** (PDFs, JSON, SQL, etc.)
- **Exporting structured summaries** to a relational DB for analytics

The core idea:
> *The moment your agent has both memory and structured knowledge access, it stops being a “chatbot” and starts behaving like a true intelligent assistant.*
